In [2]:
import os
import time
import torch
import multiprocessing as mp
from collections import Counter

from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader

print('All imports successful.')
print(f'PyTorch version: {torch.__version__}')

All imports successful.
PyTorch version: 2.10.0


## Part 1 — Non-Streaming Pipeline (Standard In-Memory Loading)

We load the full WikiText-2 training split into memory, tokenize it,
group tokens into fixed-length blocks of 128, and create a DataLoader.
This mirrors a typical small-scale training pipeline.

In [3]:
# Load dataset (non-streaming — entire dataset in RAM)
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
print(f'Number of examples in train split: {len(dataset)}')

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Number of examples in train split: 36718


In [4]:
# Initialize GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'EOS token: "{tokenizer.eos_token}" (id={tokenizer.eos_token_id})')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Vocab size: 50257
EOS token: "<|endoftext|>" (id=50256)


In [5]:
# Tokenize the full dataset using batched .map()
def tokenize_function(examples):
    return tokenizer(examples['text'], return_special_tokens_mask=False)

tokenized_ds = dataset.map(tokenize_function, batched=True, remove_columns=['text'])
print('Sample token IDs (first example, first 20 tokens):')
print(tokenized_ds[0]['input_ids'][:20])

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Sample token IDs (first example, first 20 tokens):
[]


In [6]:
# Group tokens into fixed-length blocks of 128
BLOCK_SIZE = 128

def group_texts(examples):
    concat_ids   = sum(examples['input_ids'], [])
    concat_masks = sum(examples['attention_mask'], [])
    total_len = (len(concat_ids) // BLOCK_SIZE) * BLOCK_SIZE
    concat_ids   = concat_ids[:total_len]
    concat_masks = concat_masks[:total_len]
    result_ids   = [concat_ids[i:i+BLOCK_SIZE]   for i in range(0, total_len, BLOCK_SIZE)]
    result_masks = [concat_masks[i:i+BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)]
    return {'input_ids': result_ids, 'attention_mask': result_masks}

lm_ds = tokenized_ds.map(group_texts, batched=True, batch_size=1000)
print(f'Total fixed-length LM sequences: {len(lm_ds)}')

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Total fixed-length LM sequences: 18667


In [7]:
# Create DataLoader
def collate_fn_standard(batch):
    input_ids = torch.tensor([ex['input_ids'] for ex in batch], dtype=torch.long)
    return {'input_ids': input_ids, 'labels': input_ids.clone()}

train_loader = DataLoader(lm_ds, batch_size=8, shuffle=True, collate_fn=collate_fn_standard)

# Verify one batch
batch = next(iter(train_loader))
print(f'input_ids shape : {batch["input_ids"].shape}')
print(f'labels shape    : {batch["labels"].shape}')

input_ids shape : torch.Size([8, 128])
labels shape    : torch.Size([8, 128])


## Part 2 — Token Frequency & Vocabulary Analysis

We analyse which tokens appear most in the WikiText-2 training set.
This is useful for understanding dataset characteristics before training.

In [8]:
# Count token frequencies across the entire tokenized dataset
token_counter = Counter()

for example in lm_ds:
    token_counter.update(example['input_ids'])

print(f'Unique tokens used : {len(token_counter)} / {tokenizer.vocab_size}')
print(f'Total tokens       : {sum(token_counter.values()):,}')

Unique tokens used : 36993 / 50257
Total tokens       : 2,389,376


In [9]:
# Top-20 most frequent tokens
top_20 = token_counter.most_common(20)

print(f'{"Rank":<6} {"Token ID":<12} {"Token String":<20} {"Count"}')
print('-' * 55)
for rank, (tok_id, count) in enumerate(top_20, 1):
    tok_str = repr(tokenizer.decode([tok_id]))
    print(f'{rank:<6} {tok_id:<12} {tok_str:<20} {count:,}')

Rank   Token ID     Token String         Count
-------------------------------------------------------
1      262          ' the'               113,103
2      837          ' ,'                 99,790
3      764          ' .'                 73,330
4      286          ' of'                56,865
5      290          ' and'               50,541
6      287          ' in'                39,556
7      284          ' to'                39,183
8      257          ' a'                 34,398
9      796          ' ='                 29,499
10     366          ' "'                 28,293
11     220          ' '                  23,974
12     198          '\n'                 23,696
13     2488         ' @'                 22,784
14     31           '@'                  22,781
15     373          ' was'               20,987
16     705          " '"                 18,623
17     383          ' The'               17,693
18     12           '-'                  17,007
19     82           's'         

In [10]:
# How many unique tokens cover 80% / 90% / 99% of all token occurrences?
total_tokens = sum(token_counter.values())
sorted_counts = sorted(token_counter.values(), reverse=True)

for threshold in [0.80, 0.90, 0.99]:
    cumulative, n = 0, 0
    for c in sorted_counts:
        cumulative += c
        n += 1
        if cumulative / total_tokens >= threshold:
            break
    print(f'{int(threshold*100)}% of tokens covered by top {n:>5} unique tokens '
          f'({n/tokenizer.vocab_size*100:.1f}% of vocab)')

80% of tokens covered by top  3404 unique tokens (6.8% of vocab)
90% of tokens covered by top  8118 unique tokens (16.2% of vocab)
99% of tokens covered by top 25567 unique tokens (50.9% of vocab)


## Part 3 — Streaming Pipeline with Rolling Buffer

Instead of loading everything into memory, we use HuggingFace's streaming
mode. A rolling buffer accumulates tokens across document boundaries and
yields complete fixed-length blocks — exactly how large-scale LLM
pre-training pipelines work.

In [12]:
# Load dataset in streaming mode
stream_dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train', streaming=True)
print(type(stream_dataset))  # IterableDataset — no .len(), no random access

<class 'datasets.iterable_dataset.IterableDataset'>


In [13]:
# Tokenize on-the-fly via .map()
def tokenize_streaming(examples):
    return tokenizer(examples['text'])

tokenized_stream = stream_dataset.map(tokenize_streaming, batched=True)

In [14]:
# Rolling buffer generator
def rolling_buffer_blocks(dataset_iter, block_size):
    buffer = []
    for example in dataset_iter:
        buffer.extend(example['input_ids'])
        while len(buffer) >= block_size:
            chunk = buffer[:block_size]
            buffer = buffer[block_size:]
            yield {'input_ids': chunk, 'attention_mask': [1] * block_size}
    # Yield padded remainder if any tokens are left
    if buffer:
        pad_len = block_size - len(buffer)
        yield {
            'input_ids'      : buffer + [tokenizer.pad_token_id] * pad_len,
            'attention_mask' : [1] * len(buffer) + [0] * pad_len
        }

In [15]:
# Wrap in PyTorch IterableDataset
class StreamingLMDataset(IterableDataset):
    def __init__(self, hf_iterable, block_size):
        self.hf_iterable = hf_iterable
        self.block_size  = block_size

    def __iter__(self):
        return rolling_buffer_blocks(self.hf_iterable, self.block_size)

streaming_lm_ds = StreamingLMDataset(tokenized_stream, BLOCK_SIZE)

# DataLoader
def collate_fn_streaming(batch):
    input_ids      = torch.tensor([ex['input_ids']      for ex in batch], dtype=torch.long)
    attention_mask = torch.tensor([ex['attention_mask'] for ex in batch], dtype=torch.long)
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': input_ids.clone()}

stream_loader = DataLoader(streaming_lm_ds, batch_size=8, collate_fn=collate_fn_streaming)

print('Streaming batches (first 3):')
for i, batch in enumerate(stream_loader):
    print(f'  Batch {i} -> input_ids: {batch["input_ids"].shape}')
    if i == 2:
        break

Streaming batches (first 3):
  Batch 0 -> input_ids: torch.Size([8, 128])
  Batch 1 -> input_ids: torch.Size([8, 128])
  Batch 2 -> input_ids: torch.Size([8, 128])


## Part 4 — Multi-Process Sharding with DataLoader

For large-scale training, we shard the data across multiple processes
so each worker handles a disjoint subset of the dataset. This avoids
data duplication when using multiple CPU cores or GPUs.

In [17]:
# Manual round-robin sharding
def manual_shard(dataset_iter, num_shards, process_index):
    """Yields only the examples assigned to this shard (round-robin)."""
    for idx, example in enumerate(dataset_iter):
        if idx % num_shards == process_index:
            yield example

In [18]:
# Sharded streaming IterableDataset
class ShardedStreamingLMDataset(IterableDataset):
    def __init__(self, dataset_name, dataset_config, split, tokenizer, block_size, num_shards, shard_index):
        self.dataset_name   = dataset_name
        self.dataset_config = dataset_config
        self.split          = split
        self.tokenizer      = tokenizer
        self.block_size     = block_size
        self.num_shards     = num_shards
        self.shard_index    = shard_index

    def __iter__(self):
        ds = load_dataset(self.dataset_name, self.dataset_config,
                          split=self.split, streaming=True)
        sharded = manual_shard(ds, self.num_shards, self.shard_index)
        token_stream = (
            self.tokenizer(ex['text'], add_special_tokens=False)['input_ids']
            for ex in sharded
        )
        buffer = []
        for tokens in token_stream:
            buffer.extend(tokens)
            while len(buffer) >= self.block_size:
                chunk  = buffer[:self.block_size]
                buffer = buffer[self.block_size:]
                yield {
                    'input_ids'      : torch.tensor(chunk, dtype=torch.long),
                    'attention_mask' : torch.ones(self.block_size, dtype=torch.long)
                }

In [19]:
# Worker function (runs in a subprocess)
def worker_fn(rank, world_size, results_queue, block_size, batch_size, batches_to_show):
    tok = AutoTokenizer.from_pretrained('gpt2')
    tok.pad_token = tok.eos_token

    ds = ShardedStreamingLMDataset(
        dataset_name='wikitext', dataset_config='wikitext-2-raw-v1',
        split='train', tokenizer=tok,
        block_size=block_size, num_shards=world_size, shard_index=rank
    )

    def collate(batch):
        return {
            'input_ids'      : torch.stack([ex['input_ids']      for ex in batch]),
            'attention_mask' : torch.stack([ex['attention_mask'] for ex in batch])
        }

    loader = DataLoader(ds, batch_size=batch_size, collate_fn=collate, num_workers=0)
    log = []
    for i, batch in enumerate(loader):
        msg = f'[rank {rank}] batch {i} -> shape {tuple(batch["input_ids"].shape)}'
        log.append(msg)
        if i + 1 >= batches_to_show:
            break
    results_queue.put((rank, log))

In [23]:
# Simulate multi-process sharding in a single process (Jupyter-safe)
NUM_PROCS    = 4
BATCH_SIZE   = 4
BATCHES_SHOW = 2

print('Multi-process sharding output (simulated):')

for rank in range(NUM_PROCS):
    # Each rank gets its own sharded streaming dataset
    ds = ShardedStreamingLMDataset(
        dataset_name='wikitext', dataset_config='wikitext-2-raw-v1',
        split='train', tokenizer=tokenizer,
        block_size=BLOCK_SIZE, num_shards=NUM_PROCS, shard_index=rank
    )

    def collate(batch):
        return {
            'input_ids'      : torch.stack([ex['input_ids']      for ex in batch]),
            'attention_mask' : torch.stack([ex['attention_mask'] for ex in batch])
        }

    loader = DataLoader(ds, batch_size=BATCH_SIZE, collate_fn=collate, num_workers=0)

    for i, batch in enumerate(loader):
        print(f'  [rank {rank}] batch {i} -> shape {tuple(batch["input_ids"].shape)}')
        if i + 1 >= BATCHES_SHOW:
            break

print('\nDone. Each rank processed a unique shard of the dataset.')

Multi-process sharding output (simulated):
  [rank 0] batch 0 -> shape (4, 128)
  [rank 0] batch 1 -> shape (4, 128)
  [rank 1] batch 0 -> shape (4, 128)
  [rank 1] batch 1 -> shape (4, 128)
  [rank 2] batch 0 -> shape (4, 128)
  [rank 2] batch 1 -> shape (4, 128)
  [rank 3] batch 0 -> shape (4, 128)
  [rank 3] batch 1 -> shape (4, 128)

Done. Each rank processed a unique shard of the dataset.


## Part 5 — Batch Timing & Throughput Benchmarking

We compare the throughput (tokens/second) of the non-streaming vs
streaming pipelines over the same number of batches.

In [24]:
def benchmark_loader(loader, num_batches=50, label='Loader'):
    batch_times  = []
    total_tokens = 0

    for i, batch in enumerate(loader):
        t_start = time.perf_counter()
        _ = batch['input_ids'].float().mean()  # lightweight op to touch tensor
        t_end = time.perf_counter()

        batch_times.append(t_end - t_start)
        total_tokens += batch['input_ids'].numel()

        if i + 1 >= num_batches:
            break

    total_time   = sum(batch_times)
    mean_latency = total_time / len(batch_times) * 1000  # ms
    throughput   = total_tokens / total_time              # tokens/sec

    print(f'\n=== {label} ===')
    print(f'  Batches processed  : {len(batch_times)}')
    print(f'  Total tokens       : {total_tokens:,}')
    print(f'  Total time         : {total_time:.3f} s')
    print(f'  Mean batch latency : {mean_latency:.3f} ms')
    print(f'  Throughput         : {throughput:,.0f} tokens/sec')
    return throughput

In [25]:
NUM_BENCHMARK_BATCHES = 50

# Non-streaming
tp_standard = benchmark_loader(train_loader, NUM_BENCHMARK_BATCHES, label='Non-Streaming (in-memory)')

# Streaming (fresh iterator)
stream_dataset2   = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train', streaming=True)
tokenized_stream2 = stream_dataset2.map(tokenize_streaming, batched=True)
streaming_lm_ds2  = StreamingLMDataset(tokenized_stream2, BLOCK_SIZE)
stream_loader2    = DataLoader(streaming_lm_ds2, batch_size=8, collate_fn=collate_fn_streaming)

tp_streaming = benchmark_loader(stream_loader2, NUM_BENCHMARK_BATCHES, label='Streaming (on-the-fly)')


=== Non-Streaming (in-memory) ===
  Batches processed  : 50
  Total tokens       : 51,200
  Total time         : 0.005 s
  Mean batch latency : 0.093 ms
  Throughput         : 10,986,433 tokens/sec

=== Streaming (on-the-fly) ===
  Batches processed  : 50
  Total tokens       : 51,200
  Total time         : 0.000 s
  Mean batch latency : 0.003 ms
  Throughput         : 348,791,101 tokens/sec


In [27]:
print('\n========== Throughput Summary ==========')
print(f'  Non-Streaming : {tp_standard:>10,.0f} tokens/sec')
print(f'  Streaming     : {tp_streaming:>10,.0f} tokens/sec')
faster = 'Non-Streaming' if tp_standard > tp_streaming else 'Streaming'
ratio  = max(tp_standard, tp_streaming) / min(tp_standard, tp_streaming)
print(f'\n  {faster} is {ratio:.2f}x faster for batch iteration.')
print('\nNote: Non-streaming pays an upfront cost (tokenization + grouping).')
print('      Streaming avoids that cost but tokenizes on every iteration.')


========== Throughput Summary ==========
  Non-Streaming : 10,986,433 tokens/sec
  Streaming     : 348,791,101 tokens/sec

  Streaming is 31.75x faster for batch iteration.

Note: Non-streaming pays an upfront cost (tokenization + grouping).
      Streaming avoids that cost but tokenizes on every iteration.
